<a href="https://colab.research.google.com/github/luizalmin-netizen/js-developer-pokedex/blob/main/Sistema_Banc%C3%A1rio_com_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import datetime
import json
import os

JSON_FILE = 'banco_estado.json' # Define the JSON file for persistence

class ContaBancaria:
    def __init__(self, saldo_inicial=0, limite=500, limite_saques=3):
        self.saldo = saldo_inicial
        self.limite = limite
        self.extrato = ""
        self.numero_saques = 0
        self.LIMITE_SAQUES = limite_saques
        self.last_transaction_date = datetime.date.today() # Initialize with current date
        self._carregar_estado() # Try to load state on initialization

    def _to_dict(self):
        # Convert object state to a dictionary for JSON serialization
        return {
            'saldo': self.saldo,
            'limite': self.limite,
            'extrato': self.extrato,
            'numero_saques': self.numero_saques,
            'LIMITE_SAQUES': self.LIMITE_SAQUES,
            'last_transaction_date': self.last_transaction_date.isoformat() # Convert date to string
        }

    def _from_dict(self, data):
        # Populate object from a dictionary loaded from JSON
        self.saldo = data.get('saldo', 0)
        self.limite = data.get('limite', 500)
        self.extrato = data.get('extrato', "")
        self.numero_saques = data.get('numero_saques', 0)
        self.LIMITE_SAQUES = data.get('LIMITE_SAQUES', 3)
        date_str = data.get('last_transaction_date')
        if date_str:
            self.last_transaction_date = datetime.date.fromisoformat(date_str)
        else:
            self.last_transaction_date = datetime.date.today()

    def _salvar_estado(self):
        # Save the current state to the JSON file
        try:
            with open(JSON_FILE, 'w') as f:
                json.dump(self._to_dict(), f)
            # print(f"Estado salvo em {JSON_FILE}") # Uncomment for debugging
        except Exception as e:
            print(f"Erro ao salvar estado: {e}")

    def _carregar_estado(self):
        # Load the state from the JSON file
        if os.path.exists(JSON_FILE):
            try:
                with open(JSON_FILE, 'r') as f:
                    data = json.load(f)
                    self._from_dict(data)
                print(f"Estado carregado de {JSON_FILE}")
            except Exception as e:
                print(f"Erro ao carregar estado de {JSON_FILE}: {e}. Inicializando com valores padrão.")
        else:
            print(f"Arquivo {JSON_FILE} não encontrado. Inicializando com valores padrão.")

    def _check_daily_reset(self):
        current_date = datetime.date.today()
        if current_date > self.last_transaction_date:
            self.numero_saques = 0
            self.last_transaction_date = current_date
            print("\n--- Novo dia: Limite de saques resetado ---")

    def depositar(self, valor):
        self._check_daily_reset() # Check for daily reset before any transaction
        current_date = datetime.date.today() # Get current date for transaction record
        if valor > 0:
            self.saldo += valor
            self.extrato += f"{current_date.strftime('%Y-%m-%d')} - Depósito: R$ {valor:.2f}\n"
            print(f"Depósito de R$ {valor:.2f} realizado com sucesso!")
            self._salvar_estado() # Save state after successful operation
        else:
            print("Operação falhou! O valor informado é inválido.")

    def sacar(self, valor):
        self._check_daily_reset() # Check for daily reset before any transaction
        current_date = datetime.date.today() # Get current date for transaction record

        excedeu_saldo = valor > self.saldo
        excedeu_limite = valor > self.limite
        excedeu_saques = self.numero_saques >= self.LIMITE_SAQUES

        if excedeu_saldo:
            print("Operação falhou! Você não tem saldo suficiente.")
        elif excedeu_limite:
            print("Operação falhou! O valor do saque excede o limite.")
        elif excedeu_saques:
            print("Operação falhou! Número máximo de saques diários excedido.")
        elif valor > 0:
            self.saldo -= valor
            self.extrato += f"{current_date.strftime('%Y-%m-%d')} - Saque: R$ {valor:.2f}\n"
            self.numero_saques += 1
            print(f"Saque de R$ {valor:.2f} realizado com sucesso!")
            self._salvar_estado() # Save state after successful operation
        else:
            print("Operação falhou! O valor informado é inválido.")

    def exibir_extrato(self):
        self._check_daily_reset() # Ensure daily reset check for accurate remaining saques
        current_date = datetime.date.today() # Get current date for display purposes
        print("\n================ EXTRATO ================")
        print(f"Data do extrato: {current_date.strftime('%Y-%m-%d')}\n")
        print("Não foram realizadas movimentações." if not self.extrato else self.extrato)
        print(f"\nSaldo atual: R$ {self.saldo:.2f}")
        print(f"Saques restantes para hoje: {self.LIMITE_SAQUES - self.numero_saques}")
        print("=========================================")

menu = """

[d] Depositar
[s] Sacar
[e] Extrato
[q] Sair

=> """

conta = ContaBancaria() # The __init__ method will now handle loading existing state

while True:
    opcao = input(menu)

    if opcao == "d":
        valor = float(input("Informe o valor do depósito: "))
        conta.depositar(valor)

    elif opcao == "s":
        valor = float(input("Informe o valor do saque: "))
        conta.sacar(valor)

    elif opcao == "e":
        conta.exibir_extrato()

    elif opcao == "q":
        print("Saindo do sistema. Obrigado!")
        conta._salvar_estado() # Ensure state is saved on exit
        break

    else:
        print("Operação inválida, por favor selecione novamente a operação desejada.")

Arquivo banco_estado.json não encontrado. Inicializando com valores padrão.


KeyboardInterrupt: Interrupted by user